# 04b - Export the harmonised NPP-VIIRS-like nighttime lights (Chen et al. 2021) to Google Drive

Replaces the two-sensor nightlights (DMSP 1992-2013, VIIRS 2012-) with **one consistent series** for every panel year, so no cohort needs an imputed pre-treatment nightlights mean.

* **Source:** Chen, Z., Yu, B., et al. (2021). *An extended time series (2000-2018) of global NPP-VIIRS-like nighttime light data from a cross-sensor calibration.* Earth System Science Data 13, 889-906, doi:10.5194/essd-13-889-2021 (original data: Harvard Dataverse doi:10.7910/DVN/YGIVCD). Earth Engine copy in the community catalog (sat-io, CC0): `projects/sat-io/open-datasets/npp-viirs-ntl`, extended beyond 2018.
* **Grid:** the project's own 1 km x 1 km grid asset (`grid_geometry_1km_shp`, continental Colombia only), the same asset and 64-chunk split as notebook 04, so `cell_id` matches the panel exactly.
* **Reduction:** mean over each 1 km cell at `GEE_SCALE_NTL` = 1000 m (as notebook 04's nightlights), tile scale 16.
* **Efficiency:** all years are stacked into one multi-band image, so each grid chunk is a single export task (64 tasks total instead of one per year), and only `cell_id` plus the yearly values are written (no geometry column).
* **Output:** Drive folder `gee_covariates_core_1km` (the folder notebook 04 reads), files `core_covariates_1km_chen_ntl_chunkNN.csv`, columns `cell_id`, `chen_ntl_<year>`.

The first cells only inspect the collection; tasks are submitted only if every panel year is present (`REQUIRED_YEARS`).

In [ ]:
import json
import time
from pathlib import Path
import ee

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
GEE_PROJECT = json.loads((PROJECT_ROOT / 'configs' / 'project_config.json').read_text())['gee_project']
ee.Initialize(project=GEE_PROJECT)

GRID_TAG = '1km'
GEE_GRID_ASSET_ID = f'projects/{GEE_PROJECT}/assets/grid_geometry_{GRID_TAG}_shp'
CHEN_ASSET = 'projects/sat-io/open-datasets/npp-viirs-ntl'
GEE_EXPORT_FOLDER = f'gee_covariates_core_{GRID_TAG}'
GEE_EXPORT_PREFIX = f'core_covariates_{GRID_TAG}_chen_ntl'
GEE_SCALE_NTL = 1000            # as notebook 04
GEE_TILE_SCALE = 16             # as notebook 04
N_GRID_CHUNKS = 64              # as notebook 04 (1 km grid)
GRID_CHUNK_SEED = 42            # as notebook 04
REQUIRED_YEARS = list(range(2000, 2025))   # panel 2001-2025; pre-treatment means need years before each cohort
SUBMIT_TASKS = True
print('Earth Engine project:', GEE_PROJECT)

## Inspect the collection: years, bands, resolution

In [ ]:
chen = ee.ImageCollection(CHEN_ASSET)
info = chen.limit(1).getInfo()
first = ee.Image(chen.first())
band_names = first.bandNames().getInfo()
nominal_scale = first.projection().nominalScale().getInfo()
starts = chen.aggregate_array('system:time_start').getInfo()
ids = chen.aggregate_array('system:index').getInfo()
import datetime as dt
years_available = sorted({dt.datetime.utcfromtimestamp(t / 1000).year for t in starts if t is not None})
print(f'{len(ids)} images | bands {band_names} | nominal scale {nominal_scale:.0f} m')
print('Years available:', years_available)
missing = [y for y in REQUIRED_YEARS if y not in years_available]
print('Missing panel years:', missing or 'none')
print('First image properties:', {k: v for k, v in info['features'][0]['properties'].items() if not k.startswith('system:footprint')})

## Build one multi-band image (one band per year) and submit one task per grid chunk

In [ ]:
if missing:
    raise RuntimeError(f'Chen series does not cover these panel years: {missing}. Not submitting tasks.')
BAND = band_names[0]

def year_band(y):
    img = chen.filterDate(f'{y}-01-01', f'{y + 1}-01-01').select(BAND).mean()
    return img.rename(f'chen_ntl_{y}')

chen_img = ee.Image.cat([year_band(y) for y in REQUIRED_YEARS])
out_bands = [f'chen_ntl_{y}' for y in REQUIRED_YEARS]

grid_ee = ee.FeatureCollection(GEE_GRID_ASSET_ID)
def add_grid_chunk(f):
    return f.set('grid_chunk', ee.Number(f.get('grid_random')).multiply(N_GRID_CHUNKS).floor().int())
grid_ee = grid_ee.randomColumn('grid_random', seed=GRID_CHUNK_SEED).map(add_grid_chunk)

tasks = {}
if SUBMIT_TASKS:
    for chunk in range(N_GRID_CHUNKS):
        fc = grid_ee.filter(ee.Filter.eq('grid_chunk', chunk))
        reduced = chen_img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=GEE_SCALE_NTL, tileScale=GEE_TILE_SCALE)
        desc = f'{GEE_EXPORT_PREFIX}_chunk{chunk:02d}'
        task = ee.batch.Export.table.toDrive(collection=reduced, description=desc, folder=GEE_EXPORT_FOLDER,
                                            fileNamePrefix=desc, fileFormat='CSV', selectors=['cell_id'] + out_bands)
        task.start()
        tasks[desc] = task
    print(f'Submitted {len(tasks)} export tasks to Drive folder "{GEE_EXPORT_FOLDER}".')

## Task status (re-run this cell to refresh)

In [ ]:
from collections import Counter
states = {d: t.status()['state'] for d, t in tasks.items()}
print(Counter(states.values()))
failed = {d: tasks[d].status().get('error_message') for d, s in states.items() if s == 'FAILED'}
if failed:
    print('Failed tasks:', failed)